# Benchmark 2: few-cycle compression of a 31 fs, ~1 mJ pulse

**Paper:** M. Müller, J. Buldt, H. Stark, C. Grebing, J. Limpert, *Multipass cell for high-power few-cycle compression*, Opt. Lett. (2021), [doi:10.1364/OL.425872](https://doi.org/10.1364/OL.425872), open access at [arXiv:2201.03276](https://arxiv.org/abs/2201.03276).

**Why this case:** it is the stress test of the UPPE. The spectrum grows from about 970–1090 nm to 750–1300 nm, so self-steepening, gas dispersion over a full octave-scale window and the mirror bandwidth all matter. The input is the output of benchmark 1, as in the paper (their simulation also starts from the output of the previous stage). It also runs at a high P/Pc, about 0.8, with injection into the linear eigenmode.

| | paper | used here |
|---|---|---|
| mirrors | 13 individual 1" Ag-enhanced mirrors on Si, RoC 1 m, R > 98.5 %, \|GDD\| < 10 fs² over 800–1300 nm | R = 0.985 in band, GDD = 0, R → 0.9 outside (assumption) |
| geometry | d = 1.95 m, 13 foci | `MPC(R=1, d=1.95, n_passes=13)` |
| eigenmode | waist 230 µm, 1.43 mm on mirror (radii) | computed below |
| gas | 0.5 bar Ar + 0.5 bar He | mixture: dispersions added, n₂ = n₂(Ar) + n₂(He) |
| input | 474 W at 500 kHz (0.948 mJ), 31 fs from stage 1 | `bench1_stage1_output.npz` (spectrum + phase), or a 31 fs Gaussian |
| compression | chirped mirrors −540 fs² + 21 mm fused silica | same, plus the 1 mm exit window |

**Measured results to compare with:** −20 dB span 750–1300 nm, 82 % transmission, compressed 6.9 fs with 61 % of the peak power of the 5.6 fs Fourier limit, M² < 1.2 (at 897 nm), effective homogeneity 97.5 %.

In [ ]:
import os, time, json, hashlib
import numpy as np
import matplotlib.pyplot as plt
import thunder4d as th
from thunder4d import diagnostics as dg

FS = 1e-15
plt.rcParams.update({"figure.dpi": 110})
print("thunder4d", th.__version__, "| GPU available:", th.gpu_available())

In [ ]:
# ---------------- run settings ----------------
QUICK = False          # True: 2 passes on a coarse grid, only to check that the notebook runs
DEVICE = "auto"        # "auto" (GPU if CuPy works), "gpu" or "cpu"
CPU_BUDGET_GB = 16     # memory the run may use when it runs on the CPU
PHI_MAX = 0.02         # peak nonlinear phase per step: halve it once to check convergence
MAX_SECONDS = None     # e.g. 3600: stop cleanly after ~1 h, re-run the run cell to continue
RESUME = True          # continue from the checkpoint of an identical run if there is one

In [ ]:
# ---------------- helpers (same in the three benchmark notebooks) ----------------
def device_budget_gb():
    """Memory the run may use: 80 % of the free GPU memory, or CPU_BUDGET_GB on the CPU."""
    if DEVICE != "cpu" and th.gpu_available():
        import cupy as cp
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
        return 0.8 * free / 1e9
    print(f"CPU run, budget {CPU_BUDGET_GB} GB")
    return CPU_BUDGET_GB


def pick_grid(cell, presets, budget_gb, n_fields=12, **grid_args):
    """Finest preset whose estimated memory (about n_fields x one field) fits in budget_gb."""
    for p in presets:
        g = th.suggest_grid(cell, **grid_args, **p, device="cpu")       # axes only: cheap
        need = n_fields * np.prod(g.shape) * 8 / 1e9                    # complex64
        ok = need <= budget_gb
        print(f"  {p}: {g.Nx}x{g.Ny}x{g.Nt}, ~{need:.1f} GB" + ("  <- used" if ok else ""))
        if ok:
            return th.suggest_grid(cell, **grid_args, **p, device=DEVICE), p
    raise MemoryError("no preset fits the memory budget: add a coarser preset or use a bigger GPU")


def band_mirror(R_in, lam_lo, lam_hi, R_out=0.0, order=80):
    """Intensity reflectivity R(lambda) [lambda in m]: R_in inside the band, smooth edges with their
    50 % points at lam_lo and lam_hi, R_out far outside."""
    lc, hw = 0.5 * (lam_lo + lam_hi), 0.5 * (lam_hi - lam_lo)
    def R(lam):
        u = np.abs((np.asarray(lam, float) - lc) / hw)
        return R_out + (R_in - R_out) * np.exp(-np.log(2) * u ** order)
    return R


def gdd_phase(grid, gdd_fs2=0.0, tod_fs3=0.0):
    """Spectral phase on grid.Omega (FFT order) of a GDD [fs^2] and TOD [fs^3]."""
    W = grid.Omega * FS
    return gdd_fs2 / 2 * W ** 2 + tod_fs3 / 6 * W ** 3


def beam_pixels(Aw, rel=1e-4):
    """(npix, Nt) host array of the pixels that carry the beam (fluence > rel x max)."""
    A = th.to_host(Aw)
    F = np.sum(np.abs(A) ** 2, axis=2)
    return A[F > rel * F.max()]


def _pad(E, up):
    """Zero-pad spectra (FFT order, last axis) by `up` -> time step dt/up."""
    if up == 1:
        return E
    N = E.shape[-1]
    h = (N + 1) // 2
    out = np.zeros(E.shape[:-1] + (N * up,), complex)
    out[..., :h] = E[..., :h]
    out[..., N * up - (N - h):] = E[..., h:]
    return out


def integrated_power(grid, E_pix, phase=None, up=4):
    """Spatially integrated P(t) [W] of E_pix (npix, Nt) after the spectral phase `phase`
    (array on grid.Omega). Returns sorted t [fs] and P [W], time step dt/up."""
    E = E_pix if phase is None else E_pix * np.exp(1j * phase)[None, :]
    A = grid.hfft.ift_t(_pad(E, up)) * up
    P = np.sum(np.abs(A) ** 2, axis=0) * grid.dA
    t = np.fft.fftfreq(grid.Nt * up, 1.0 / (grid.Nt * grid.dt))
    return np.fft.fftshift(t) / FS, np.fft.fftshift(P)


def fourier_limit(grid, E_pix, up=4):
    """Fourier limit of the spatially integrated spectrum (the 'Fourier limit' quoted in papers)."""
    S = np.sum(np.abs(E_pix) ** 2, axis=0)
    return integrated_power(grid, np.sqrt(S)[None, :], None, up)


def fwhm(x, y):
    """FWHM of the main peak (linear interpolation)."""
    y = np.asarray(y, float)
    i = int(np.argmax(y)); h = y[i] / 2
    l, r = i, i
    while l > 0 and y[l] > h: l -= 1
    while r < len(y) - 1 and y[r] > h: r += 1
    xl = np.interp(h, [y[l], y[l + 1]], [x[l], x[l + 1]])
    xr = np.interp(h, [y[r], y[r - 1]], [x[r], x[r - 1]])
    return xr - xl


def span(lam, S, level_db=-20):
    """(lambda_min, lambda_max) [nm] of the outermost points above level_db relative to the peak."""
    ok = np.nonzero(S / S.max() >= 10 ** (level_db / 10))[0]
    return lam[ok[0]] * 1e9, lam[ok[-1]] * 1e9


def homogeneity_1d(S, rel=np.exp(-2)):
    """Spectral homogeneity as defined by Weitenberg et al. (Opt. Express 25, 20502, 2017) and used in
    the papers: V(x) = (int sqrt(S S_ref))^2 / (int S int S_ref), S_ref = spectrum at the intensity
    maximum. S[x, lambda]. Returns V(x) and its intensity-weighted mean inside the 1/e^2 width."""
    prof = S.sum(axis=1)
    S0 = S[np.argmax(prof)]
    V = np.sum(np.sqrt(S * S0[None, :]), axis=1) ** 2 / np.maximum(S.sum(axis=1) * S0.sum(), 1e-300)
    inside = prof > rel * prof.max()
    return V, float(np.sum(V[inside] * prof[inside]) / np.sum(prof[inside]))


def load_digitized(path):
    """Optional measured curve digitised from a paper figure: CSV with 2 columns (x, y)."""
    if not os.path.exists(path):
        print(f"(no {path}: digitise the figure, e.g. with WebPlotDigitizer, to overlay the measurement)")
        return None
    d = np.loadtxt(path, delimiter=",")
    d = d[np.argsort(d[:, 0])]
    return d[:, 0], d[:, 1] / d[:, 1].max()


def table(rows):
    """rows: (quantity, simulation, paper) -> aligned text table."""
    print(f"{'quantity':42s}{'simulation':>16s}{'paper':>22s}")
    print("-" * 80)
    for q, s, p in rows:
        print(f"{q:42s}{s:>16s}{p:>22s}")


def run_tag(params, grid):
    """Short fingerprint of the parameters and grid, used to name the checkpoint."""
    s = json.dumps({k: str(v) for k, v in sorted(params.items())}) + str(grid.shape) + str(PHI_MAX) + str(QUICK)
    return hashlib.md5(s.encode()).hexdigest()[:8]

## 1. Parameters from the paper

In [ ]:
P = dict(
    lambda0=1030e-9,
    energy=0.948e-3,         # paper: 474 W at 500 kHz entering the second cell
    stage1_file="bench1_stage1_output.npz",   # output of notebook 1; used if it exists
    fallback_fwhm=31e-15,    # if not: transform-limited Gaussian of the stage-1 duration
    R=1.0, d=1.95, n_passes=13,               # paper
    p_ar=0.5, p_he=0.5,      # paper: 0.5 bar Ar + 0.5 bar He
    n2_ar_1bar=1.0e-23,      # same argon n2 as notebook 1
    mirror_R=0.985,          # paper: R > 98.5 % over 800-1300 nm
    band=(780e-9, 1320e-9),  # assumption: 50 % edges 20 nm outside the specified band
    R_out=0.9,               # assumption: reflectivity far outside the band
    cm_gdd=-540.0,           # paper: chirped mirrors
    fs_mm=21.0 + 1.0,        # paper: 21 mm fused silica + 1 mm exit window
    lambda_min=650e-9, lambda_max=1450e-9,
    window_fwhm=80e-15,      # sets the time window (x time_window_factor)
    injection="linear",
)
REF = dict(w0_um=230, wm_um=1430, span="750-1300", T=0.82, tl_fs=5.6, comp_fs=6.9, peak=0.61,
           M2="< 1.2", V="97.5 %")
RUN_DIR = "runs_bench2"
os.makedirs(RUN_DIR, exist_ok=True)

## 2. Gas mixture, cell and eigenmode

In [ ]:
ar = th.Material("Ar", P["p_ar"], n2_1bar=P["n2_ar_1bar"])
he = th.Material("He", P["p_he"])
gas = th.Material("Ar+He", n_func=lambda w: 1 + (ar.n(w) - 1) + (he.n(w) - 1), n2=ar.n2 + he.n2)
cell = th.MPC(R=P["R"], d=P["d"], medium=gas, n_passes=2 if QUICK else P["n_passes"],
              mirror_reflectivity=band_mirror(P["mirror_R"], *P["band"], R_out=P["R_out"]))
lin = cell.linear_mode(P["lambda0"])
print(cell.report())
print(f"Pc = {gas.critical_power(P['lambda0'])/1e9:.1f} GW, GVD = {gas.gvd(P['lambda0']):.1f} fs^2/m")
print(f"linear eigenmode: waist {lin['w0']*1e6:.0f} um (paper {REF['w0_um']}), "
      f"mirror {lin['w_mirror']*1e6:.0f} um (paper {REF['wm_um']})")

## 3. Grid

In [ ]:
PRESETS = ([dict(box_factor=4, points_per_waist=3, time_window_factor=4)] if QUICK else
           [dict(box_factor=8, points_per_waist=8, time_window_factor=8),
            dict(box_factor=7, points_per_waist=7, time_window_factor=7),
            dict(box_factor=6, points_per_waist=6, time_window_factor=7)])
grid, preset = pick_grid(cell, PRESETS, device_budget_gb(), lambda0=P["lambda0"],
                         lambda_min=P["lambda_min"], lambda_max=P["lambda_max"], pulse_fwhm=P["window_fwhm"])
print(grid.describe())

## 4. Input: the output of benchmark 1
The spatially integrated spectrum and the on-axis spectral phase after the stage-1 compressor are put on a Gaussian beam at the eigenmode waist.

In [ ]:
if os.path.exists(P["stage1_file"]):
    d1 = np.load(P["stage1_file"])
    if bool(d1["quick"]):
        print("WARNING: the stage-1 file comes from a QUICK run of notebook 1")
    E = th.measured_spectrum(grid, d1["lam_nm"], d1["S"], phase_rad=d1["phase"])
    src = "stage-1 output (notebook 1)"
else:
    E = th.gaussian_spectrum(grid, fwhm_fs=P["fallback_fwhm"] / FS)
    src = f"Gaussian {P['fallback_fwhm']/FS:.0f} fs (notebook 1 output not found)"

probe = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(lin["w0"])), P["energy"])
try:
    mm = cell.match(probe, verbose=True)
except ValueError as err:                    # P >= Pc: no Kerr-matched mode
    print(err); mm = None
w0_ref = lin["w0"] if (P["injection"] == "linear" or mm is None) else mm["w0"]
pulse = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(w0_ref), chromatic_size="eigenmode"), P["energy"])
t, Pt = integrated_power(grid, beam_pixels(pulse.Aw))
print(f"input: {src} | {fwhm(t, Pt):.1f} fs, peak {Pt.max()/1e9:.1f} GW, "
      f"P/Pc = {Pt.max()/gas.critical_power(P['lambda0']):.2f}")

## 5. Run

In [ ]:
sim = th.MPCSimulation(pulse, cell, phi_max=PHI_MAX, mode_reference=w0_ref, near_field=True)

In [ ]:
tag = run_tag(P, grid)
ckpt = os.path.join(RUN_DIR, f"checkpoint_{tag}.pkl")
if RESUME and os.path.exists(ckpt):
    sim.resume(ckpt)
    print(f"resumed from {ckpt} at pass {sim.passes_done}")
if not sim.done:
    t0 = time.time()
    sim.run(max_seconds=MAX_SECONDS)
    sim.checkpoint(ckpt)
    print(f"{time.time() - t0:.0f} s for this call")
print("finished" if sim.done else "time budget reached: run this cell again to continue")

## 6. Evolution along the cell

In [ ]:
h = sim.history
npass = np.array(h["pass"])
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2), layout="constrained")
ax[0].plot(npass, np.array(h["energy"]) / h["energy"][0], "o-"); ax[0].set_title("transmission")
ax[1].plot(npass, h["B"], "o-"); ax[1].set_title("B-integral, on-axis peak (rad)")
ax[2].plot(npass, np.array(h["w_peak_x"]) * 1e6, "o-", label="x")
ax[2].plot(npass, np.array(h["w_peak_y"]) * 1e6, "s-", ms=3, label="y")
ax[2].axhline(w0_ref * 1e6, color="k", ls=":", label="injected waist")
ax[2].set_title("beam radius at the cell centre (um)"); ax[2].legend(fontsize=8)
ax[3].plot(npass, h["eta00_total"], "o-", label="HG00 fraction")
ax[3].plot(npass, h["V_near"], "s-", label="homogeneity (2D, near field)")
ax[3].plot(npass, h["M2x_mean"], "^-", label="M2 x"); ax[3].legend(fontsize=8)
for a in ax:
    a.set_xlabel("pass")

## 7. Output spectrum
Optional overlay: digitise Fig. 3 of the paper into `data/muller2021_spectrum.csv` (nm, linear intensity; the figure is in dB, so convert with 10^(dB/10)).

In [ ]:
XLIM_NM = (650, 1450)
lam, S = dg.total_spectrum(grid, sim.Aw)
S_in = sim.history["spectrum"][0]
meas = load_digitized("data/muller2021_spectrum.csv")
fig, ax = plt.subplots(figsize=(8, 3.6), layout="constrained")
ax.plot(sim.lam * 1e9, 10 * np.log10(S_in / S_in.max() + 1e-12), "k:", label="input")
ax.plot(lam * 1e9, 10 * np.log10(S / S.max() + 1e-12), label="simulation")
if meas is not None:
    ax.plot(meas[0], 10 * np.log10(meas[1] + 1e-12), label="measured (digitised)")
ax.axvspan(800, 1300, color="0.9", zorder=0, label="mirror band (paper)")
ax.set_xlim(*XLIM_NM); ax.set_ylim(-40, 3); ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("dB"); ax.legend(fontsize=8)
lo, hi = span(lam, S, -20)
print(f"-20 dB span: {lo:.0f}-{hi:.0f} nm (paper {REF['span']} nm)")

## 8. Compression as in the experiment
Chirped mirrors (−540 fs²) plus 22 mm of fused silica (Malitson Sellmeier). The constant and linear parts of the glass phase (a delay) are removed.

In [ ]:
def n_fs(lam_um):
    """Fused silica, Malitson 1965."""
    l2 = lam_um ** 2
    return np.sqrt(1 + 0.6961663 * l2 / (l2 - 0.0684043 ** 2) + 0.4079426 * l2 / (l2 - 0.1162414 ** 2)
                   + 0.8974794 * l2 / (l2 - 9.896161 ** 2))

w = np.where(grid.valid, grid.omega, grid.omega0)
phi_glass = n_fs(2 * np.pi * th.C0 / w * 1e6) * w / th.C0 * P["fs_mm"] * 1e-3
sel = np.abs(grid.Omega) < 0.2 * np.abs(grid.Omega).max()
c1, c0 = np.polyfit(grid.Omega[sel], phi_glass[sel], 1)
phi_glass = (phi_glass - c1 * grid.Omega - c0) * grid.valid
phi_exp = gdd_phase(grid, P["cm_gdd"]) + phi_glass

Epix = beam_pixels(sim.Aw)
UP = 8
t_tl, P_tl = fourier_limit(grid, Epix, UP)
t_e, P_e = integrated_power(grid, Epix, phi_exp, UP)
best = dg.compress(grid, sim.Aw, gdd_bounds_fs2=(-1500, 500), n_scan=81)
t_b, P_b = integrated_power(grid, Epix, gdd_phase(grid, best["gdd_fs2"]), UP)

fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
ax.plot(t_tl, P_tl / P_tl.max(), "k:", label=f"Fourier limit {fwhm(t_tl, P_tl):.1f} fs")
ax.plot(t_e, P_e / P_tl.max(), label=f"-540 fs$^2$ + {P['fs_mm']:.0f} mm FS: {fwhm(t_e, P_e):.1f} fs, "
        f"{P_e.max()/P_tl.max()*100:.0f} %")
ax.plot(t_b, P_b / P_tl.max(), "--", label=f"best pure GDD {best['gdd_fs2']:.0f} fs$^2$: {fwhm(t_b, P_b):.1f} fs")
ax.set_xlim(-40, 40); ax.set_xlabel("time (fs)"); ax.set_ylabel("power / Fourier limit"); ax.legend(fontsize=8)

## 9. Output beam: spatio-spectral homogeneity

In [ ]:
# near field (collimated output beam), along x: what the papers' imaging spectrometers measure
theta, lam_t, S_th = dg.spectrogram_theta_lambda(grid, sim.Aw, per_lambda=False)
V_x, V_mean = homogeneity_1d(S_th)
prof = S_th.sum(axis=1)
inside = prof > np.exp(-2) * prof.max()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
ax[0].pcolormesh(lam_t * 1e9, theta * 1e3, S_th / S_th.max(), shading="auto", cmap="inferno")
ax[0].set_xlim(*XLIM_NM); ax[0].set_ylim(theta[inside].min() * 2e3, theta[inside].max() * 2e3)
ax[0].set_xlabel("wavelength (nm)"); ax[0].set_ylabel("angle (mrad)"); ax[0].set_title("output S(theta_x, lambda)")
ax[1].plot(theta * 1e3, prof / prof.max(), "k", label="profile")
ax[1].plot(theta[inside] * 1e3, V_x[inside], "g", label="homogeneity V")
ax[1].set_xlim(ax[0].get_ylim()); ax[1].set_ylim(0, 1.05); ax[1].legend(fontsize=8)
ax[1].set_xlabel("angle (mrad)"); ax[1].set_title(f"weighted homogeneity (x): {V_mean*100:.1f} %")

## 10. Summary

In [ ]:
Tcell = sim.history["energy"][-1] / sim.history["energy"][0]
table([
    ("waist / mirror spot (um, 1/e^2 radius)", f"{lin['w0']*1e6:.0f} / {lin['w_mirror']*1e6:.0f}", f"{REF['w0_um']} / {REF['wm_um']}"),
    ("-20 dB span (nm)",                       f"{lo:.0f}-{hi:.0f}", REF["span"]),
    ("transmission of the cell",               f"{Tcell*100:.1f} %", f"{REF['T']*100:.0f} %"),
    ("Fourier limit (fs)",                     f"{fwhm(t_tl, P_tl):.1f}", f"{REF['tl_fs']}"),
    ("compressed as in the experiment (fs)",   f"{fwhm(t_e, P_e):.1f}", f"{REF['comp_fs']}"),
    ("peak power vs Fourier limit",            f"{P_e.max()/P_tl.max()*100:.0f} %", f"{REF['peak']*100:.0f} %"),
    ("M2 x / y (spectrally weighted)",         f"{sim.history['M2x_mean'][-1]:.2f} / {sim.history['M2y_mean'][-1]:.2f}", REF["M2"]),
    ("homogeneity, weighted (x)",              f"{V_mean*100:.1f} %", REF["V"]),
    ("B-integral, on-axis peak (rad)",         f"{sim.solver.B:.1f}", "-"),
])
sim.save(os.path.join(RUN_DIR, "run.npz"), include_field=False)

## Checks before trusting the comparison
* **Convergence:** this case is the most demanding. Check `PHI_MAX = 0.01` and a finer grid; also check that the spectrum stays well inside the spectral window (the solver tapers the nonlinear source at the window edges).
* **Mirror model:** the reflectivity outside 800–1300 nm and the residual GDD (|GDD| < 10 fs²) are assumptions; the blue edge of the spectrum is sensitive to them.
* **Input:** the stage-1 output from notebook 1 is idealised (no pedestal from the real laser, a perfect compressor). Comparing with the Gaussian fallback shows how much the input matters.
* **Measurement:** the paper's M² is taken at the CCD-weighted wavelength (897 nm), and its homogeneity is a lower bound (low signal above 1.1 µm).